In [16]:
# Importamos yfinance con el alias "yf", que es la convención de la comunidad
import yfinance as yf

In [15]:
# Descargamos dos meses de precios diarios del SPY (el ETF que replica el S&P 500).
# - start/end: rango de fechas en formato AAAA-MM-DD (la fecha "end" NO se incluye)
# - auto_adjust=True: entrega precios AJUSTADOS por dividendos y splits (lo correcto en análisis)
# - progress=False: oculta la barra de progreso para que la salida quede limpia
spy = yf.download("SPY", start="2024-01-01", end="2024-03-01", auto_adjust=True, progress=False)

# .head() muestra las primeras 5 filas: una forma rápida de "mirar" los datos
spy.head()

Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2024-01-02,457.672699,458.660396,455.581141,457.198236,123623700
2024-01-03,453.935028,456.258971,453.334679,455.523045,103585900
2024-01-04,452.472900,456.036282,452.250178,453.460568,84232200
2024-01-05,453.092621,455.532757,451.649816,452.676224,86118900
2024-01-08,459.560913,459.706154,453.460529,453.586415,74879100


In [14]:
# .shape devuelve (número de filas, número de columnas)
print("Forma:", spy.shape)

# .columns muestra los nombres de las columnas
print("Columnas:", spy.columns.tolist())

# .index es la "columna de fechas" que identifica cada fila
print("Primera fecha:", spy.index.min())
print("Última fecha:", spy.index.max())

Forma: (41, 5)
Columnas: [('Close', 'SPY'), ('High', 'SPY'), ('Low', 'SPY'), ('Open', 'SPY'), ('Volume', 'SPY')]
Primera fecha: 2024-01-02 00:00:00
Última fecha: 2024-02-29 00:00:00


In [13]:
# Descargamos lo mismo pero SIN ajustar, para comparar.
# Con auto_adjust=False, yfinance entrega dos columnas de cierre:
#   "Close"     -> precio crudo, tal como se transó ese día
#   "Adj Close" -> precio ajustado por dividendos
spy_crudo = yf.download("SPY", start="2024-01-01", end="2024-03-01", auto_adjust=False, progress=False)

# Comparamos las dos columnas lado a lado
spy_crudo[["Close", "Adj Close"]].head()

Price,Close,Adj Close
Ticker,SPY,SPY
Date,,
2024-01-02,472.649994,457.672699
2024-01-03,468.790009,453.935028
2024-01-04,467.279999,452.472900
2024-01-05,467.920013,453.092621
2024-01-08,474.600006,459.560913


In [12]:
# Descargamos Entel y el IPSA desde 2014 (el mismo periodo de tu trabajo de Finanzas II).
# - El sufijo ".SN" indica que el activo cotiza en la Bolsa de Santiago.
# - El "^" indica que es un índice, no una acción que se pueda comprar directamente.
# - ["Close"] se queda solo con los precios de cierre (ajustados, por auto_adjust=True).
chile = yf.download(["ENTEL.SN", "^IPSA"], start="2014-01-01", auto_adjust=True, progress=False)["Close"]

print("Forma:", chile.shape)
print("\nPrimeras filas:")
print(chile.head())
print("\nÚltimas filas:")
print(chile.tail())

# .isna().sum() cuenta cuántos valores FALTANTES (NaN = "Not a Number") hay en cada columna.
# Es la primera revisión de calidad que se hace sobre cualquier dato nuevo.
print("\nValores faltantes por columna:")
print(chile.isna().sum())

$^IPSA: possibly delisted; no timezone found

1 Failed download:
['^IPSA']: possibly delisted; no timezone found


Forma: (3186, 2)

Primeras filas:
Ticker         ENTEL.SN  ^IPSA
Date                          
2014-01-02  3165.422852    NaN
2014-01-03  3170.539551    NaN
2014-01-06  3189.635742    NaN
2014-01-07  3150.914307    NaN
2014-01-08  3153.252441    NaN

Últimas filas:
Ticker      ENTEL.SN  ^IPSA
Date                       
2026-10-13    3977.0    NaN
2026-10-14    3977.0    NaN
2026-10-15    3977.0    NaN
2026-10-16    3977.0    NaN
2026-10-19    4249.0    NaN

Valores faltantes por columna:
Ticker
ENTEL.SN       0
^IPSA       3186
dtype: int64


In [17]:
# yf.Ticker() crea un objeto con toda la información de UN activo.
# .dividends devuelve el historial de dividendos que Yahoo tiene registrado.
entel = yf.Ticker("ENTEL.SN")
div = entel.dividends

print("Cantidad de dividendos registrados:", len(div))
print(div.tail(10))  # los 10 más recientes

Cantidad de dividendos registrados: 24
Date
2021-10-28 09:30:00-03:00    700.0000
2022-04-25 09:30:00-04:00    200.0000
2022-05-26 09:30:00-04:00    470.0000
2022-11-28 09:30:00-03:00    100.0000
2023-05-15 09:30:00-04:00     94.0000
2024-04-25 09:30:00-04:00    235.4419
2024-12-23 09:30:00-03:00     50.0000
2025-04-25 09:30:00-04:00    129.0000
2025-11-17 09:30:00-03:00    115.0000
2026-04-27 09:30:00-04:00    140.0000
Name: Dividends, dtype: float64


In [19]:
# Probamos varios tickers candidatos para representar el mercado chileno.
# - "^IPSA" y "^SPIPSA": posibles nombres del índice en Yahoo
# - "ECH": ETF de iShares que replica las acciones chilenas, transado en EE.UU. (en dólares)
candidatos = ["^IPSA", "^SPIPSA", "ECH"]

for ticker in candidatos:
    datos = yf.download(ticker, start="2014-01-01", auto_adjust=True, progress=False)
    if datos.empty:
        print(f"{ticker:10} -> ❌ sin datos")
    else:
        print(f"{ticker:10} -> ✅ {len(datos)} filas, desde {datos.index.min().date()} hasta {datos.index.max().date()}")

$^IPSA: possibly delisted; no timezone found

1 Failed download:
['^IPSA']: possibly delisted; no timezone found
$^SPIPSA: possibly delisted; no timezone found

1 Failed download:
['^SPIPSA']: possibly delisted; no timezone found


^IPSA      -> ❌ sin datos
^SPIPSA    -> ❌ sin datos
ECH        -> ✅ 3218 filas, desde 2014-01-02 hasta 2026-10-19


In [20]:
# Descargamos Entel SIN ajustar para ver "Close" (crudo) y "Adj Close" (ajustado) lado a lado
entel_raw = yf.download("ENTEL.SN", start="2014-01-01", auto_adjust=False, progress=False)

# .droplevel("Ticker", axis=1) elimina el segundo nivel de encabezado (el MultiIndex),
# porque con un solo activo no lo necesitamos y simplifica la tabla
comp = entel_raw[["Close", "Adj Close"]].droplevel("Ticker", axis=1)

# Creamos una columna nueva: si el ajuste funciona, este ratio debería ser
# MENOR que 1 en fechas antiguas (el ajustado "descuenta" los dividendos futuros)
comp["ratio"] = comp["Adj Close"] / comp["Close"]

# .iloc[[...]] selecciona filas por POSICIÓN: la primera (0), la del medio y la última (-1)
print(comp.iloc[[0, len(comp) // 2, -1]])

print("\nPrimer dividendo registrado:", div.index.min().date())

Price             Close    Adj Close     ratio
Date                                          
2014-01-02  7012.766602  3165.424072  0.451380
2020-05-27  5198.000000  2539.441162  0.488542
2026-10-19  4249.000000  4249.000000  1.000000

Primer dividendo registrado: 2010-12-07


## 📋 Conclusiones de la exploración

**Fuente evaluada:** Yahoo Finance vía `yfinance` 1.7.0

### Hallazgos
1. **Calendario:** 2 meses de SPY = 41 filas (44 días hábiles − 3 feriados de EE.UU.). Se usarán 252 días/año para anualizar.
2. **Precios ajustados:** `auto_adjust=True` corrige por dividendos y splits. En SPY la diferencia fue de ~3,2% en 2024.
3. **IPSA no disponible:** `^IPSA` y `^SPIPSA` no tienen datos. Se usará **ECH** (iShares MSCI Chile, en USD) como proxy del mercado chileno.
4. **Entel (`ENTEL.SN`):** el ajuste por dividendos sí funciona (ratio Adj/Close ≈ 0,45 en 2014; 24 dividendos registrados desde 2010).
5. **Calidad de datos:**
   - Precio repetido (3977) del 13 al 16 de octubre de 2026 → posible dato congelado.
   - La última fila corresponde al día en curso → precio no definitivo.

### Decisiones para el ETL
- Usar precios ajustados (`auto_adjust=True`).
- Excluir el día de mercado en curso.
- Validar automáticamente valores faltantes y precios repetidos.
- Pendiente (Fase 4): convertir ECH a pesos chilenos para calcular el beta de Entel.